# XAI Attribution Viewer

Minimal viewer for XAI outputs stored in `.npz` files.

**Controls:** Model, Patient ID, Slice, Show Zones  
**Displays (per channel T2W / ADC / HBV):** Original image · Image + labels · Saliency · Ablation CAM · Input ablation bar · Occlusion

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, ToggleButton

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
XAI_ROOT = Path(".").resolve() / "results" / "xai"

MODEL_XAI_DIRS = {
    "nnunet":     XAI_ROOT / "nnunet",
    "umamba_mtl": XAI_ROOT / "umamba_mtl",
    "swin_unetr": XAI_ROOT / "swin_unetr",
}

MODEL_OPTIONS = [
    ("nnUNet",     "nnunet"),
    ("U-MambaMTL", "umamba_mtl"),
    ("SwinUNETR",  "swin_unetr"),
]

CMAP     = "hot"
CH_NAMES = ["T2W", "ADC", "HBV"]

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def find_files(model: str):
    d = MODEL_XAI_DIRS.get(model)
    if d is None or not d.exists():
        return []
    files = []
    for f in sorted(d.glob("fold_*/*.npz")):
        try:
            tmp = np.load(f, allow_pickle=True)
            _ = tmp["image"]
            files.append(f)
        except Exception:
            print(f"  [skipped corrupt file: {f.name}]")
    return files


def load(npz_path):
    d = np.load(npz_path, allow_pickle=True)

    def spatial(key):
        return d[key] if key in d and d[key].ndim > 1 else None

    def vec(key):
        return d[key] if key in d and d[key].size > 0 else None

    return {
        "image":          d["image"],
        "label":          spatial("label"),
        "prediction":     spatial("prediction"),
        "saliency":       spatial("saliency"),
        "occlusion":      spatial("occlusion"),
        "ablation":       spatial("ablation"),
        "input_ablation": vec("input_ablation"),
        "zones":          spatial("zones"),
        "channels":       list(d["channels"]),
        "case_id":        str(d["case_id"]),
        "fold":           int(d["fold"]),
    }


def norm_mri(vol):
    mn, mx = vol.min(), vol.max()
    return (vol - mn) / (mx - mn) if mx > mn else np.zeros_like(vol, dtype=np.float32)


def norm_global(arr):
    mx = arr.max()
    return arr / mx if mx > 0 else arr.copy()


def show_overlay(ax, mri_sl, attr_sl, title, cmap=CMAP, alpha=0.5):
    ax.imshow(mri_sl, cmap="gray", vmin=0, vmax=1, aspect="equal")
    ax.imshow(attr_sl, cmap=cmap, vmin=0, vmax=1, aspect="equal", alpha=alpha)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def show_mri(ax, mri_sl, title):
    ax.imshow(mri_sl, cmap="gray", vmin=0, vmax=1, aspect="equal")
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def overlay_zones(ax, zone_sl, alpha=0.45):
    """Overlay TZ (blue) and PZ (green).  zone values: 1=PZ, 2=TZ."""
    rgba = np.zeros((*zone_sl.shape, 4), dtype=np.float32)
    rgba[zone_sl == 2] = [0.15, 0.35, 1.00, alpha]  # TZ — blue
    rgba[zone_sl == 1] = [0.10, 0.85, 0.20, alpha]  # PZ — green
    ax.imshow(rgba, aspect="equal")


def plot_ablation_bar(ax, weights, highlight_ch, title):
    w = np.clip(weights, 0, None)
    total = w.sum()
    w_norm = w / total if total > 0 else w.copy()
    colors = ["tab:orange" if i == highlight_ch else "tab:blue" for i in range(len(w_norm))]
    ax.bar(CH_NAMES, w_norm, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_ylim(0, max(1.05, float(w_norm.max()) * 1.2))
    for i, v in enumerate(w_norm):
        ax.text(i, float(v) + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.set_ylabel("Norm. importance", fontsize=8)
    ax.tick_params(labelsize=8)


_TZ_PATCH = mpatches.Patch(facecolor=(0.15, 0.35, 1.00, 0.7), label="TZ (Transition Zone)")
_PZ_PATCH = mpatches.Patch(facecolor=(0.10, 0.85, 0.20, 0.7), label="PZ (Peripheral Zone)")

print("Setup complete.")
for label, key in MODEL_OPTIONS:
    files = find_files(key)
    print(f"  {label:12s}: {len(files)} case(s)")

Setup complete.
  nnUNet      : 14 case(s)
  U-MambaMTL  : 0 case(s)
  SwinUNETR   : 0 case(s)


In [15]:
# ---------------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------------
_model_w = Dropdown(
    options=MODEL_OPTIONS, value="nnunet",
    description="Model:", style={"description_width": "initial"},
)

_init_files = find_files(_model_w.value)
_case_w = Dropdown(
    options=[(f.stem, f) for f in _init_files] if _init_files else [("(no files)", None)],
    description="Case:",
    style={"description_width": "initial"},
    layout={"width": "420px"},
)
_slice_w = IntSlider(min=0, max=0, step=1, value=0, description="Slice:")
_zones_w = ToggleButton(
    value=True, description="Show Zones", button_style="info",
    tooltip="Overlay TZ (blue) and PZ (green) on all image panels",
)
_zone_alpha_w = FloatSlider(
    min=0.0, max=1.0, step=0.05, value=0.45,
    description="Zone α:", readout_format=".2f",
    style={"description_width": "initial"},
)
_xai_alpha_w = FloatSlider(
    min=0.0, max=1.0, step=0.05, value=0.5,
    description="XAI α:", readout_format=".2f",
    style={"description_width": "initial"},
)


def _refresh_slice(npz_path):
    if npz_path is None:
        return
    d = load(npz_path)
    n = d["image"].shape[1]
    _slice_w.max = n - 1
    _slice_w.value = min(_slice_w.value, n - 1)


def _on_model_change(change):
    files = find_files(change["new"])
    _case_w.options = [(f.stem, f) for f in files] if files else [("(no files)", None)]
    if files:
        _refresh_slice(_case_w.value)


def _on_case_change(change):
    _refresh_slice(change["new"])


_model_w.observe(_on_model_change, names="value")
_case_w.observe(_on_case_change, names="value")
_refresh_slice(_case_w.value)

# ---------------------------------------------------------------------------
# Main viewer
# ---------------------------------------------------------------------------
@interact(model=_model_w, npz_path=_case_w, z=_slice_w, show_zones=_zones_w,
          zone_alpha=_zone_alpha_w, xai_alpha=_xai_alpha_w)
def view(model, npz_path, z, show_zones, zone_alpha, xai_alpha):
    if npz_path is None:
        print(f"No files found for model '{model}'.")
        return

    data = load(npz_path)
    n_slices = data["image"].shape[1]
    ch_names = [c.upper() for c in data["channels"]]
    n_ch = len(ch_names)

    has_label    = data["label"] is not None
    has_saliency = data["saliency"] is not None
    has_ablation = data["ablation"] is not None
    has_iabl     = data["input_ablation"] is not None
    has_occ      = data["occlusion"] is not None
    has_zones    = show_zones and data["zones"] is not None

    # Build row list in display order
    rows = ["Original"]
    if has_label:    rows.append("Image + Labels")
    if has_saliency: rows.append("Saliency")
    if has_ablation: rows.append("Ablation CAM")
    if has_iabl:     rows.append("Input Ablation")
    if has_occ:      rows.append("Occlusion")
    n_rows = len(rows)

    fig, axes = plt.subplots(
        n_rows, n_ch,
        figsize=(4.5 * n_ch, 4.2 * n_rows),
        squeeze=False,
    )

    sal = norm_global(data["saliency"]) if has_saliency else None
    occ = norm_global(data["occlusion"]) if has_occ else None
    abl = norm_global(data["ablation"]) if has_ablation else None
    zones_sl = data["zones"][z] if has_zones else None

    for col, ch in enumerate(ch_names):
        mri = norm_mri(data["image"][col])  # (D, H, W)
        sl  = mri[z]

        row = 0

        # ── Original image ──────────────────────────────────────────────────
        show_mri(axes[row, col], sl, f"Original  {ch}  z={z}/{n_slices-1}")
        if has_zones:
            overlay_zones(axes[row, col], zones_sl, alpha=zone_alpha)
        row += 1

        # ── Image + Labels ───────────────────────────────────────────────────
        if has_label:
            show_overlay(axes[row, col], sl, data["label"][0, z],
                         f"Image + Labels  {ch}", cmap="Reds", alpha=min(xai_alpha + 0.2, 1.0))
            if has_zones:
                overlay_zones(axes[row, col], zones_sl, alpha=zone_alpha)
            row += 1

        # ── Saliency ─────────────────────────────────────────────────────────
        if has_saliency:
            show_overlay(axes[row, col], sl, sal[col, z], f"Saliency  {ch}", alpha=xai_alpha)
            if has_zones:
                overlay_zones(axes[row, col], zones_sl, alpha=zone_alpha)
            row += 1

        # ── Ablation CAM ─────────────────────────────────────────────────────
        if has_ablation:
            show_overlay(axes[row, col], sl, abl[0, z], f"Ablation CAM  {ch}", alpha=xai_alpha)
            if has_zones:
                overlay_zones(axes[row, col], zones_sl, alpha=zone_alpha)
            row += 1

        # ── Input Ablation bar ───────────────────────────────────────────────
        if has_iabl:
            plot_ablation_bar(
                axes[row, col], data["input_ablation"],
                highlight_ch=col,
                title=f"Input Ablation  (↑ {ch})",
            )
            row += 1

        # ── Occlusion ────────────────────────────────────────────────────────
        if has_occ:
            show_overlay(axes[row, col], sl, occ[col, z], f"Occlusion  {ch}", alpha=xai_alpha)
            if has_zones:
                overlay_zones(axes[row, col], zones_sl, alpha=zone_alpha)

    # Row labels on left
    for r, label in enumerate(rows):
        axes[r, 0].set_ylabel(label, fontsize=11, labelpad=8)

    plt.suptitle(
        f"[{model}]  case {data['case_id']}  fold {data['fold']}",
        fontsize=13, y=1.01,
    )

    if show_zones:
        fig.legend(
            handles=[_TZ_PATCH, _PZ_PATCH],
            loc="lower right", fontsize=9, framealpha=0.8,
        )

    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Model:', options=(('nnUNet', 'nnunet'), ('U-MambaMTL', 'umamba_mtl…